# FTS — Are our master times telling the truth?

**What this notebook is.** A plain-language walkthrough of how we check the
theoretical times in the ERP against what actually happens on the shop floor,
why each methodological choice was made, and what we have found so far.

**Keep it current.** This notebook is the source — edit it directly in Jupyter
or PyCharm. Every time an operation is added, or the flow changes, re-run all
cells. Every number below is computed live from the data files — nothing is
typed in by hand — so a stale notebook is a broken notebook, not a slightly
out-of-date one.

## 1. The question

Every routing step carries a **standard**: how long the job *should* take. In the
ERP that standard is stored as `RUN_MASTER`. For the operations we analyse it is
a **rate** — pieces per hour — and the theoretical time follows from it:

$$\text{HOURS\_MASTER} = \frac{\text{quantity}}{\text{RUN\_MASTER}}$$

So a `RUN_MASTER` of 100 on an order of 50 pieces implies half an hour.

Against that we have `HOURS_WORKED`: the sum of the labour tickets people
actually clocked. The two numbers come from completely different places — one is
an engineering assumption, the other is a record of events.

**The question is simple to state and awkward to answer:** for a given part at a
given routing step, is the gap between the two big enough, and consistent
enough, that the standard should be changed?

"Awkward" because a single work order tells you almost nothing. Jobs vary. What
we need is evidence that the gap is *systematic* rather than noise — which is a
hypothesis-testing problem.

## 2. Where the data comes from

`fts.sql` joins labour tickets to work orders, operations, parts and shop
resources, and aggregates to one row per:

> work order × routing step × labour-ticket type × resource

We read the exported extract. One important detail about the grain: the query
already restricts to **4-digit operation sequences**, which is how the business
distinguishes routings rebuilt during the master-data overhaul from the older
ones that preceded it.

That restriction is the single biggest reason these figures do not match the
operational performance dashboard, and it is the one that looks most like data
being discarded. The overhaul renumbered routing steps from 2 and 3 digits to 4
as it rebuilt them, so a shorter sequence belongs to a routing that no longer
exists -
there is no `RUN_MASTER` at sequence 20 to correct, because sequence 20 is gone.
Those hours are real work and belong on a performance dashboard; they are inert
on a list of values to edit.

Nothing below can show you this, because the 2-digit rows are not in the extract
by construction. The README carries the external check, with the query to
re-run it and the one condition that would overturn it.

In [ ]:
import numpy as np, pandas as pd
from scipy import stats
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 50)

FY_START = "2025-10-01"   # current fiscal year
MIN_WO   = 6              # minimum work orders to run a test
ALPHA    = 0.05           # false-discovery rate
MIN_EFFECT = 0.10         # smallest deviation worth acting on
MIN_HOURS  = 20           # smallest hours impact worth acting on

INK, MUTED, GRID = "#26231f", "#6a655e", "#e3e0da"
COOL, WARM       = "#3B6FB6", "#C4622D"   # below vs above standard

plt.rcParams.update({
    "figure.dpi": 120, "font.size": 9, "axes.edgecolor": GRID,
    "axes.labelcolor": MUTED, "text.color": INK, "xtick.color": MUTED,
    "ytick.color": MUTED, "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.color": GRID, "grid.linewidth": 0.6,
})

raw = pd.read_parquet("fts.parquet")
for col in ["FAMILY","PART_ID","OPERATION","LT_TYPE","RUN_TYPE_MASTER","ANALYTIC_SECTION","WORK_ORDER"]:
    raw[col] = raw[col].str.strip()
raw["LAST_TRANSACTION_DATE"] = pd.to_datetime(raw["LAST_TRANSACTION_DATE"], errors="coerce")

print(f"{len(raw):,} rows | {raw.WORK_ORDER.nunique():,} work orders | "
      f"{raw.PART_ID.nunique():,} parts | {raw.OPERATION.nunique()} operations")
print(f"dates {raw.LAST_TRANSACTION_DATE.min():%Y-%m-%d} to {raw.LAST_TRANSACTION_DATE.max():%Y-%m-%d}")

## 3. What we analyse, and what we deliberately leave out

Five exclusions. Each is a decision with a reason, not a convenience.

**Setup tickets.** A setup's theoretical time comes from `SETUP_HRS`, not
`RUN_MASTER`, so a correction factor has nowhere to go *in this pipeline* — the
arithmetic in §12 inverts, because a duration multiplies where a rate divides.
There is a second reason to keep it separate: `SETUP_HRS` is barely a standard.
Fifteen distinct values cover the entire extract, and within each crew size one
value takes the overwhelming majority — it is "half an hour times the crew",
applied regardless of the job. Testing that would measure distance from a
placeholder rather than whether a standard is right.

**`HRS/LOAD`.** This expresses the standard as a *duration per load* rather
than a rate, and charges whole loads: the arithmetic that turns a correction
factor into a new standard multiplies instead of dividing. Mixing it in would
silently corrupt every suggestion, so it is analysed separately by
`fts_load_view.py` — see §12.

**Rework (`S RIL%`).** Unplanned by nature. Across the whole extract these rows
carry thousands of worked hours against essentially **zero** master hours: there
is no standard to test against, and nor should there be.

**Steps no longer in the routing.** `MASTER_OPS_ROWID` joins each work-order
operation back to the master routing. Where it comes back empty there is no
master line to correct, so the row is dropped.

Why it is empty varies, and only one cause is well understood. Most of it is
revision deletion — one part here went from 41 sequences at revision 8 to 23 at
revision 9, and **every one of the 25 deleted sequences lacks a master row while
all 17 survivors have one**. But a quarter of the excluded hours sits on parts
that have *never been revised*, where deletion cannot be the explanation; those
parts do carry master rows on other steps, so the master routing is found and
simply holds fewer operations than production books.

**The exclusion is correct either way, which is why the remainder is left
unexplained.** With no master line there is nothing an update could target, so
the rule fails safe: it does nothing rather than something wrong. Chasing the
cause further would change no suggestion.

**Anything before the current fiscal year.** Standards changed during the
overhaul; older work orders describe a world that no longer exists.

In [ ]:
raw["IS_REWORK"] = raw.OPERATION.str.upper().str.startswith("S RIL")
raw["NO_MASTER"] = raw.MASTER_OPS_ROWID.isna()
in_scope = ((raw.LT_TYPE == "Run") & (raw.RUN_TYPE_MASTER == "PCS/HR")
            & ~raw.IS_REWORK & ~raw.NO_MASTER
            & (raw.LAST_TRANSACTION_DATE >= pd.Timestamp(FY_START)))

funnel = pd.DataFrame([
    ("All rows in the extract", len(raw), raw.HOURS_WORKED.sum()),
    ("- prior fiscal year", (raw.LAST_TRANSACTION_DATE < pd.Timestamp(FY_START)).sum(),
     raw.loc[raw.LAST_TRANSACTION_DATE < pd.Timestamp(FY_START), "HOURS_WORKED"].sum()),
    ("- rework", int(raw.IS_REWORK.sum()), raw.loc[raw.IS_REWORK, "HOURS_WORKED"].sum()),
    ("- step not in the current routing", int(raw.NO_MASTER.sum()),
     raw.loc[raw.NO_MASTER, "HOURS_WORKED"].sum()),
    ("- setup tickets", int((raw.LT_TYPE == "Setup").sum()),
     raw.loc[raw.LT_TYPE == "Setup", "HOURS_WORKED"].sum()),
    ("- HRS/LOAD", int((raw.RUN_TYPE_MASTER != "PCS/HR").sum()),
     raw.loc[raw.RUN_TYPE_MASTER != "PCS/HR", "HOURS_WORKED"].sum()),
    ("= IN SCOPE (PCS/HR run, this FY)", int(in_scope.sum()), raw.loc[in_scope, "HOURS_WORKED"].sum()),
], columns=["", "rows", "hours worked"])
funnel["hours worked"] = funnel["hours worked"].round(0)

# the categories overlap, so the deductions do not sum - each line is
# "how many rows this rule alone removes"
rw = raw[raw.IS_REWORK]
print(f"rework: {rw.HOURS_WORKED.sum():,.0f} hours worked against "
      f"{rw.HOURS_MASTER.sum():,.1f} hours of standard")
print(f"no master row: {raw.NO_MASTER.sum():,} rows, "
      f"{raw.NO_MASTER.mean():.1%} of the extract, "
      f"{raw.loc[raw.NO_MASTER & raw.IS_REWORK].shape[0] / raw.NO_MASTER.sum():.0%} of them rework\n")
funnel

A final filter inside the scope: rows where `RUN_MASTER` is 0, or where either
hours figure is 0. Those are not testable.

**The zero is a decision, not a gap** — and that is the whole reason we make no
suggestion for them. Whole operations sit at zero across every part and
every family: warehousing, document control, picking, dispatch, certificates.
Work nobody set out to time. More sits in family+sequence groups
that are entirely at zero, the same signature at smaller scale. An omission
would be patchy; this is uniform.

The distinction matters more than it looks. Proposing a standard where somebody
decided there should be none is not a correction, it is inventing policy — and
this analysis has no standing to do that. What survives the reasoning is a
handful of groups where sibling parts *do* carry a standard, which is
genuinely patchy and an order of magnitude below the materiality gate.

The cell below sizes the excluded population.

In [ ]:
scoped   = raw[in_scope]
no_std   = scoped[scoped.RUN_MASTER == 0]
testable = scoped[(scoped.RUN_MASTER > 0) & (scoped.HOURS_MASTER > 0) & (scoped.HOURS_WORKED > 0)]

print(f"in scope        {len(scoped):>7,} rows   {scoped.HOURS_WORKED.sum():>10,.0f} h")
print(f"  no standard   {len(no_std):>7,} rows   {no_std.HOURS_WORKED.sum():>10,.0f} h   <- excluded, see below")
print(f"  testable      {len(testable):>7,} rows   {testable.HOURS_WORKED.sum():>10,.0f} h")

# Is the zero deliberate? An omission would be patchy - some parts of an
# operation standardised, others not. A policy is uniform.
run = raw[(raw.LT_TYPE == "Run") & ~raw.IS_REWORK & ~raw.NO_MASTER
          & (raw.LAST_TRANSACTION_DATE >= pd.Timestamp(FY_START))]
by_op = run.groupby("OPERATION").RUN_MASTER.agg(
    rows="size", zero=lambda s: (s == 0).sum())
whole = by_op[(by_op.zero == by_op.rows)]
print(f"\noperations that are 100% RUN_MASTER = 0 : {len(whole)}")
print("   ", ", ".join(sorted(whole.index)))

grp = run.groupby(["OPERATION", "FAMILY", "PART_TYPE", "OPERATION_SEQUENCE"]).RUN_MASTER.agg(
    n="size", zero=lambda s: (s == 0).sum())
patchy = grp[(grp.zero > 0) & (grp.zero < grp.n)]
uniform = grp[(grp.zero == grp.n) & (grp.zero > 0)]
z = run[run.RUN_MASTER == 0].set_index(["OPERATION", "FAMILY", "PART_TYPE", "OPERATION_SEQUENCE"])
print(f"\nfamily+sequence groups entirely at zero : {len(uniform):>4}   "
      f"{z[z.index.isin(uniform.index)].HOURS_WORKED.sum():>7,.0f} h   deliberate")
print(f"groups where siblings DO have a standard: {len(patchy):>4}   "
      f"{z[z.index.isin(patchy.index)].HOURS_WORKED.sum():>7,.0f} h   patchy, below the gate")

## 4. The measurement: a ratio, and why we take its logarithm

For each work order we compute

$$r = \frac{\text{HOURS\_WORKED}}{\text{HOURS\_MASTER}}$$

Above 1 means the job took longer than allowed; below 1 means it was quicker.
The ratio is **dimensionless**, which is what makes everything else possible: a
60-piece order and a 600-piece order become directly comparable, and so do two
parts whose standards are 6 pcs/hr and 90 pcs/hr.

The alternative — testing the raw hours difference — fails badly, because the
difference scales with order size. One large order would dominate twenty small
ones purely by being large.

**Why the logarithm.** On a plain scale the ratio is asymmetric around its
neutral point. Twice as long is 2.0, which sits 1.0 above 1; half as long is 0.5,
only 0.5 below. Equally wrong, unequal distances — so overruns would always
outweigh underruns and every average would be biased upward. Taking logs fixes
it exactly: ln(2) = +0.69 and ln(0.5) = −0.69. The neutral point becomes 0, and
"the standard is right" becomes the clean null hypothesis **centre = 0**.

In [ ]:
wo_all = (testable
    .groupby(["OPERATION","FAMILY","PART_ID","OPERATION_SEQUENCE","WORK_ORDER"], dropna=False)
    .agg(RUN_MASTER=("RUN_MASTER","max"), LAST_DATE=("LAST_TRANSACTION_DATE","max"),
         TOTAL_QTY=("TOTAL_QTY","sum"), HOURS_WORKED=("HOURS_WORKED","sum"),
         HOURS_MASTER=("HOURS_MASTER","sum"))
    .reset_index())
wo_all["RATIO"] = wo_all.HOURS_WORKED / wo_all.HOURS_MASTER
wo_all["LOG_RATIO"] = np.log(wo_all.RATIO)

fig, ax = plt.subplots(1, 2, figsize=(9.4, 3.1))
for a, series, title, xlab in [
    (ax[0], wo_all.RATIO.clip(upper=4), "Ratio — skewed, bounded at zero", "worked / master"),
    (ax[1], wo_all.LOG_RATIO.clip(-2, 2), "Log-ratio — symmetric around 0", "ln(worked / master)")]:
    a.hist(series, bins=70, color=COOL, alpha=.85, edgecolor="none")
    a.axvline(1 if a is ax[0] else 0, color=INK, lw=1.2, ls="--")
    a.set_title(title, fontsize=9.5, color=INK, loc="left"); a.set_xlabel(xlab)
    a.set_yticks([]); a.grid(axis="x", alpha=.6); a.grid(axis="y", visible=False)
ax[0].set_ylabel("work orders", color=MUTED)
fig.suptitle("Dashed line = the standard is exactly right", fontsize=8.5, color=MUTED, y=1.04, x=.11, ha="left")
plt.tight_layout(); plt.show()

print(f"{len(wo_all):,} work-order observations   "
      f"median ratio {np.exp(wo_all.LOG_RATIO.median()):.3f}   "
      f"skew of log-ratio {wo_all.LOG_RATIO.skew():.2f}")

The left panel is why we do not test the ratio directly, and the right panel is
what we test instead.

Note the log-ratio still has a **long left tail** — a run of work orders where a
quantity was booked but almost no hours were. Those are reporting artefacts, not
superhuman speed, and §7 deals with them. That tail is also the reason our
primary test is a rank test rather than a t-test.

## 5. What counts as one "thing" to test

This choice matters more than the choice of test, and it is the one people get
wrong. Two decisions, in opposite directions: one *splits* the data, the other
refuses to.

**Split on the operation sequence.** The same operation name recurs at several
points of one routing, and it is a different job each time — a polish before
assembly is not the polish after it. So `S FRESATURA` at sequence 1200 and
`S FRESATURA` at sequence 1800 are two units, tested separately, each getting
its own suggestion. They are never pooled.

The routing itself is the evidence for this. Only a minority of part+operation
pairs appear at more than one sequence, but among those that do, most carry a
*different* `RUN_MASTER` at each one — somebody deliberately set two different
standards because the two jobs are not the same. The examples below show what
gets lost by merging: the same part and the same operation name, running at
two points of one routing, with standards that differ by a factor of two and
performance that differs just as much.

**Split on the part type.** A family holds both the semi-finished article
(`Articolo non di vendita`) and the finished one (`Articolo di vendita`) — in
general the `M.193` and `M.393` versions of the same product. Where they share a
routing step those are two different physical jobs: one part is half made, the
other is complete. Pooling them averages the two.

Only the groups where both types share a routing step are affected, and the
two types mostly run different operations at different sequences anyway — the
second cell below counts them. Splitting does have a cost: a group testable
when pooled can split into two halves that are each below the minimum, and it
then goes untested rather than judged on a blend of two different jobs. The
failure it prevents is the quiet kind. When the two halves
are wrong in opposite directions, the pooled ratio can land on 1.0 and the
group is reported as clean; one family's `S SEGARE` step did exactly that
when the split was introduced in August 2026. A wrong number invites a
challenge. A blend that lands on 1.0 produces no row at all.

The type is read from the `PART_TYPE` field rather than inferred from the part
number, because the number is not a reliable guide: the same cell counts the
`M.193` rows that are in fact sale articles, and checks that the `M.393`
convention still holds.

**Do not split on the work order revision.** `WO_REV` is bumped for *any* edit
anywhere in the routing — a note changed on some unrelated step is enough. It
says almost nothing about whether *this* step's time changed. Keying on it would
shatter every sample for no information gain. Section 6 covers what we key on
instead.

In [ ]:
u = wo_all.groupby(["PART_ID","OPERATION","OPERATION_SEQUENCE"]).size()
po = wo_all.groupby(["PART_ID","OPERATION"]).OPERATION_SEQUENCE.nunique()
seq_rm = (wo_all.groupby(["PART_ID","OPERATION","OPERATION_SEQUENCE"]).RUN_MASTER.median()
          .groupby(["PART_ID","OPERATION"]).nunique())

print(f"part+operation pairs                        : {len(po):>6,}")
print(f"  ...appearing at more than one sequence    : {(po>1).sum():>6,}  ({(po>1).mean():.0%})")
print(f"  ...of those, a different RUN_MASTER per seq: {(seq_rm[po>1]>1).sum():>6,}  ({(seq_rm[po>1]>1).mean():.0%})")
print(f"\nunit of analysis = PART_ID x OPERATION_SEQUENCE x OPERATION -> {len(u):,} units")

# Concrete cases: one part, one operation name, two points in the routing, and a
# different standard at each. One example per operation, best-evidenced first.
split = seq_rm[(po > 1) & (seq_rm > 1)]
by_wo = wo_all.groupby(["PART_ID","OPERATION","OPERATION_SEQUENCE"]).size()

picks, seen = [], set()
for k in sorted(split.index, key=lambda k: -by_wo.loc[k].min()):
    if k[1] not in seen:                      # one example per operation
        picks.append(k); seen.add(k[1])
    if len(picks) == 3:
        break

ex = (wo_all.set_index(["PART_ID","OPERATION"]).loc[picks].reset_index()
        .groupby(["PART_ID","OPERATION","OPERATION_SEQUENCE"])
        .agg(work_orders=("WORK_ORDER","nunique"), RUN_MASTER=("RUN_MASTER","median"),
             hours_worked=("HOURS_WORKED","sum"), hours_master=("HOURS_MASTER","sum")))
ex["ratio"] = (ex.hours_worked / ex.hours_master).round(2)

print("\nsame part, same operation name, different point in the routing:")
print(ex.round(1).to_string())

In [ ]:
# Part type: the same family x operation x sequence groups the pipeline tests,
# with and without PART_TYPE in the key. Pooled is what the key would give
# without it; the halves are what the pipeline actually tests.
K = ["FAMILY", "OPERATION", "OPERATION_SEQUENCE"]
pooled = testable.groupby(K).agg(WO=("WORK_ORDER", "nunique"), TYPES=("PART_TYPE", "nunique"))
halves = testable.groupby(K + ["PART_TYPE"]).WORK_ORDER.nunique()
best   = halves.groupby(level=[0, 1, 2]).max().reindex(pooled.index)
lost   = int(((pooled.WO >= MIN_WO) & (best < MIN_WO)).sum())

print(f"family groups, pooled without PART_TYPE     : {len(pooled):>6,}")
print(f"  ...mixing both types                      : {(pooled.TYPES > 1).sum():>6,}  "
      f"({(pooled.TYPES > 1).mean():.1%})")
print(f"testable ({MIN_WO}+ work orders), pooled           : {(pooled.WO >= MIN_WO).sum():>6,}")
print(f"testable once split                         : {(halves >= MIN_WO).sum():>6,}")
print(f"  testable pooled, but in neither half      : {lost:>6,}   <- what the split costs")

# The part number is not the type: M.193 is not always semi-finished. The
# M.393 / M.395 convention is checked here, not assumed.
pid  = raw.PART_ID.astype(str)
sale = raw.PART_TYPE == "Articolo di vendita"
m193 = int((pid.str.startswith("M.193") & sale).sum())
off  = int((pid.str.match(r"M\.39[35]") & ~sale).sum())
print(f"\nM.193 rows that are sale articles           : {m193:>6,}")
print(f"M.393 / M.395 rows that are not             : {off:>6,}   "
      + ("<- the convention holds" if off == 0 else "<- the convention is BROKEN, reword section 5"))

## 6. When the standard itself changed mid-year

Section 5 said we ignore the work order revision. That raises the obvious
question: if `WO_REV` moved from 8 to 9 at the same sequence, do those work
orders get pooled together?

**Yes — provided `RUN_MASTER` did not move with it.** The revision is not what
we track; the standard is. A part running under revision 8 and revision 9 with
the same `RUN_MASTER` throughout is one continuous body of evidence about one
standard, and splitting it would only shrink the sample.

What we do react to is the standard *itself* changing. If someone edited
`RUN_MASTER` during the period, the older work orders describe a value that no
longer exists. Pooling them does not merely add noise — it drags the estimate
back toward the number that was already rejected, which is worse than useless
when engineering has just corrected something.

**So the rule keys on the master value, not the revision.** Three outcomes:

| Situation | What we do | `BASIS` |
|---|---|---|
| The standard never moved | pool everything, whatever the revision did | `single master` |
| It moved, and there are ≥ `MIN_WO` work orders on the new value | test *only* those | `current master only` |
| It moved, and there are too few on the new value | make no suggestion — report as **watch** | `insufficient data on current master` |

That third row is the conservative one. We could pool the old orders back in to
reach the sample size, but then we would be "correcting" the new standard using
evidence about the old one. Better to say we cannot judge it yet.

The table below counts how often the two things move independently — and the
top-left/bottom-right corners are the point of this section.

In [ ]:
# Rebuild the per-unit picture including WO_REV, which wo_all does not carry.
uw = (testable.groupby(["OPERATION","PART_ID","OPERATION_SEQUENCE","WORK_ORDER"], dropna=False)
      .agg(RUN_MASTER=("RUN_MASTER","max"), WO_REV=("WO_REV","first"),
           LAST_DATE=("LAST_TRANSACTION_DATE","max")).reset_index())

def unit_basis(g):
    """Mirror of test_group() in fts_analysis.py, for illustration."""
    cur     = g.loc[g.LAST_DATE.idxmax(), "RUN_MASTER"]   # the value in force now
    on_cur  = int((g.RUN_MASTER == cur).sum())
    changed = g.RUN_MASTER.nunique() > 1
    return pd.Series({
        "n_wo": len(g), "n_revs": g.WO_REV.nunique(),
        "n_masters": g.RUN_MASTER.nunique(), "on_current": on_cur,
        "BASIS": ("single master"                       if not changed        else
                  "current master only"                 if on_cur >= MIN_WO   else
                  "insufficient data on current master")})

units = (uw.groupby(["OPERATION","PART_ID","OPERATION_SEQUENCE"])
           .apply(unit_basis, include_groups=False).reset_index())
units = units[units.n_wo >= MIN_WO]

print(f"units with at least {MIN_WO} work orders: {len(units):,}\n")
print("the two things move independently:")
print(pd.crosstab(units.n_revs > 1, units.n_masters > 1,
                  rownames=["revision changed"], colnames=["RUN_MASTER changed"]).to_string())

print("\nwhich means:")
b = (units.BASIS.value_counts().rename("units").to_frame()
       .assign(share=lambda d: (d.units / d.units.sum()).map("{:.1%}".format)))
print(b.to_string())

Read the crosstab corners:

- **Revision changed, standard did not** — the large off-diagonal cell. These
  are exactly the units that would have been needlessly split if we had keyed
  on `WO_REV`. Every one of them pools.
- **Standard changed, revision did not** — a smaller cell, and a mildly
  surprising one: `RUN_MASTER` was edited without the routing revision moving.
  It is the same signature as a manual database edit, and it is the reason
  keying on the revision would not even be *safe*, let alone efficient — a
  revision-based rule would silently pool two different standards.

The overwhelming majority of units sit on `single master` and are unaffected by
any of this. The machinery exists for the minority, and it fails safe: when in
doubt, the unit is reported and left alone.

## 7. Outliers, and the one thing we protect against

Labour-ticket discipline is imperfect. A work order booked with a full quantity
but a few minutes of clocked time produces a ratio near zero and is a data
problem, not a fast job.

We compute the **median absolute deviation** of the log-ratios and set aside
observations more than 3 MADs from the median. MAD is used rather than standard
deviation precisely because the standard deviation is itself inflated by the
outliers it is supposed to find.

**Important:** the filter affects the *estimate*, never the *test*. The
hypothesis test runs on every work order. If we trimmed first and then tested,
we would be manufacturing significance. The workbook carries both figures —
`RATIO_HOURS_WORKED_TO_HOURS_MASTER` divides exactly the two hours columns
beside it, and `RATIO_OUTLIERS_REMOVED` is the trimmed version that drives the
suggested value.

### The case MAD cannot handle: token bookings

Some tickets carry a flat **0.01 h whatever the quantity** — one piece or five
hundred, the same hundredth of an hour. That is an entry made to close the step,
not a measurement.

Trimming cannot rescue these, and the reason is worth being precise about. MAD
removes points *far from the centre*. Where the tokens are a **majority** of a
group they **are** the centre, and the filter then works in reverse: it trims
the real work orders away as outliers. The centre is a value that was never
measured, the spread around it is tiny, and both tests read a perfectly
consistent signal — the degenerate case looks like the strongest evidence in the
set.

So they are removed **before** testing, at `<= 0.02 h`, rather than trimmed
during it. This is the one filter that touches the test as well as the estimate,
and it is justified on different grounds: a booking that recorded no duration is
not an extreme observation of the quantity we are measuring, it is an absence of
one. The setup analysis has always done this — there the identical booking means
a setup that was never performed.

Nothing is capped or imputed in their place; there is no time inside them to
substitute. Removing them hands the decision to the `MIN_WO` floor, which is
where it belongs: a group either still has six real work orders or it leaves as
*insufficient data*. The cell below shows the mechanism on the worst case in the
data.

In [ ]:
TOKEN_HOURS = 0.02

tok = wo_all.HOURS_WORKED <= TOKEN_HOURS
print(f"token bookings: {tok.sum():,} of {len(wo_all):,} work orders "
      f"({tok.mean():.1%}), {wo_all.loc[tok,'HOURS_WORKED'].sum():.1f} h booked "
      f"on {wo_all.loc[tok,'TOTAL_QTY'].sum():,} pieces")
print(f"quantity on those work orders ranges {wo_all.loc[tok,'TOTAL_QTY'].min():,} "
      f"to {wo_all.loc[tok,'TOTAL_QTY'].max():,} - the booking does not move with it\n")

# the worst affected group, and what MAD does to it
g = (wo_all.assign(TOK=tok).groupby(["OPERATION","FAMILY","OPERATION_SEQUENCE"])
     .agg(n=("TOK","size"), ntok=("TOK","sum")))
op, fam, seq = g[g.n >= MIN_WO].ntok.idxmax()
ex = wo_all[(wo_all.OPERATION == op) & (wo_all.FAMILY == fam)
            & (wo_all.OPERATION_SEQUENCE == seq)]

lr = ex.LOG_RATIO.to_numpy()
med = np.median(lr)
mad = np.median(np.abs(lr - med)) * 1.4826
keep = np.abs(lr - med) <= 3 * mad if mad > 0 else np.ones(len(lr), bool)
real = (ex.HOURS_WORKED > TOKEN_HOURS).to_numpy()

print(f"{op}  ·  sequence {seq}")
print(f"  work orders                    {len(ex):>6,}")
print(f"  ...token bookings              {(~real).sum():>6,}")
print(f"  median log-ratio               {med:>6.3f}   -> ratio {np.exp(med):.4f}")
print(f"  MAD                            {mad:>6.3f}")
print(f"  trimmed away by the MAD filter {(~keep).sum():>6,}   "
      f"of which real work orders: {(~keep & real).sum()}")
print(f"\n  pooled ratio, all work orders  "
      f"{ex.HOURS_WORKED.sum()/ex.HOURS_MASTER.sum():>6.4f}")
print(f"  real work orders left          {real.sum():>6,}   "
      f"{'below' if real.sum() < MIN_WO else 'at or above'} the floor of {MIN_WO}"
      f"  -> {'no suggestion' if real.sum() < MIN_WO else 'testable'}")

# From here on the notebook works on the same population the scripts do.
wo_all = wo_all[~tok].reset_index(drop=True)
print(f"\nwo_all is now {len(wo_all):,} work orders - the population every "
      f"section below tests")

## 8. The test

The null hypothesis is **"the standard is right"** — the log-ratios should
scatter around 0 with no systematic lean. We run two tests and require both to
agree.

**Wilcoxon signed-rank — the robust one.** It discards the actual values and
keeps only their order. Rank every work order by how far it sits from zero
ignoring direction, then add up the ranks of the ones that ran long and the
ranks of the ones that ran short. If the standard were right the two sums would
be roughly equal. The p-value is the probability of a split at least as lopsided
arising by chance. Because it uses ranks, the most extreme observation counts as
"the biggest" and nothing more — a single mis-clocked ticket cannot swing it.

**One-sample t-test on the log-ratios — the informative one.** It uses the
actual values, so it is more sensitive but also more fragile. Its real
contribution is the **confidence interval**, which is the genuinely useful
output: not "significant" but *"the true ratio lies between 1.03 and 1.41"*.

Worked example below on a real unit.

In [ ]:
# Pick a small unit with a mix of over- and under-runs, so the ranking is
# visible. Chosen from the data rather than hardcoded, so this cell keeps
# working as operations are added.
cand = (wo_all.groupby(["OPERATION", "PART_ID", "OPERATION_SEQUENCE"])
              .agg(n=("LOG_RATIO", "size"), over=("LOG_RATIO", lambda s: (s > 0).sum())))
cand = cand[cand.n.between(8, 10) & cand.over.between(cand.n * .6, cand.n * .85)]
op, part, seq = cand.sort_values("n", ascending=False).index[0]

ex = wo_all[(wo_all.OPERATION == op) & (wo_all.PART_ID == part)
            & (wo_all.OPERATION_SEQUENCE == seq)].sort_values("LAST_DATE")
print(f"{op}  ·  sequence {seq}  ·  {len(ex)} work orders\n")

show = ex[["TOTAL_QTY", "HOURS_MASTER", "HOURS_WORKED", "RATIO", "LOG_RATIO"]].copy()
show["|log|_rank"] = show.LOG_RATIO.abs().rank().astype(int)
show["ran"] = np.where(show.LOG_RATIO > 0, "long", "short")
print(show.round(3).to_string(index=False))

lr = ex.LOG_RATIO.values
w_long = show.loc[show.ran == "long", "|log|_rank"].sum()
w_short = show.loc[show.ran == "short", "|log|_rank"].sum()
wil = stats.wilcoxon(lr); tt = stats.ttest_1samp(lr, 0)
lo, hi = tt.confidence_interval(1 - ALPHA)

print(f"\nrank sum, ran long   : {w_long}")
print(f"rank sum, ran short  : {w_short}   <- balanced would be ~{(w_long+w_short)/2:.0f} each")
print(f"Wilcoxon  p = {wil.pvalue:.4f}")
print(f"t-test    p = {tt.pvalue:.4f}   95% CI on the ratio [{np.exp(lo):.3f}, {np.exp(hi):.3f}]")
print(f"pooled ratio = {ex.HOURS_WORKED.sum():.1f} / {ex.HOURS_MASTER.sum():.1f} = "
      f"{ex.HOURS_WORKED.sum()/ex.HOURS_MASTER.sum():.3f}")

### Which test controls what — and one subtlety worth knowing

Multiplicity control runs on the **t-test** p-value, and Wilcoxon acts as a
robustness gate that must independently agree at raw p < 0.05.

That split is not arbitrary. Wilcoxon is a *discrete* test: with a small sample
there are only so many ways the ranks can fall, so only certain p-values are
reachable. With 7 work orders the smallest p it can **ever** return is 0.0156,
no matter how extreme the miss. After correcting for multiple testing that
blocks genuine findings for reasons of arithmetic granularity rather than
evidence — we hit exactly this, with real units at ratio 1.48 being discarded.
The t-test p-value is continuous and has no such floor.

In [ ]:
floor = pd.DataFrame({"n work orders": range(4, 11)})
floor["smallest p Wilcoxon can return"] = [stats.wilcoxon(np.full(n, .5)).pvalue for n in floor["n work orders"]]
floor["can it ever reach 0.05?"] = np.where(floor["smallest p Wilcoxon can return"] < .05, "yes", "NO")
floor.round(4)

This is also why `MIN_WO = 6` is a hard floor rather than a preference: below 6
work orders, a two-sided rank test cannot produce a significant result even if
every single order falls on the same side of the standard.

## 9. Testing many things at once

Running one test at a 5% threshold means a 5% chance of a false alarm. Running
sixty of them means expecting three false alarms by luck alone.

We apply the **Benjamini–Hochberg** false-discovery-rate adjustment, which
converts each p-value into a **q-value**. The guarantee is easy to state: among
everything we flag as significant, no more than 5% are expected to be false
positives. A raw p of 0.03 therefore does not automatically survive — whether it
does depends on how many other tests ran alongside it.

## 10. From "statistically different" to "worth changing"

These are not the same thing, and conflating them is how analyses like this
generate unusable output.

With enough work orders the test will detect a 2% deviation. Nobody should edit
a routing over 2%. So on top of significance we apply two **judgement** gates —
and they are business decisions, not statistical ones:

- the deviation must be at least **10%**
- the hours impact must be at least **20 hours** over the period

The second gate needs care. It was calibrated on operations averaging ~90 hours
per part+sequence. On a thinly-spread operation like penetrant testing —
~5 hours per unit — almost nothing clears an absolute 20-hour bar even when the
standard is wrong by a factor of eight. **That is why the absolute-hours gate is
applied at the level the change is actually made** (see §11): for a thin
operation that is the family, not the part.

## 11. Two levels: the part, and the family

Most parts simply do not run often enough to be judged alone. The median part
has a handful of work orders in the year. So we test at two levels.

**Part level** — for parts with enough work orders of their own. Specific, and
the strongest evidence when available.

**Family level** — pooling every part of a family at the same routing step,
including all the ones too thin to judge individually. Before computing the
family figure we **remove any part already flagged at part level**, so the two
lists never touch the same part and the family number is an honest statement
about everything that is left.

**Which level dominates is a property of the operation, and it is readable from
the data before you look at any result.** Check whether `RUN_MASTER` is uniform
within a family: if one value was stamped across every part, the standard *is* a
family-level object and should be corrected there. If each part carries its own
value, somebody did real standard-setting per part and the part-level number is
the meaningful one.

### Which value actually gets applied

A part flagged individually has two candidate corrections — its own, and its
family's. Preferring the part looks obviously right and usually is not.

Take a part measuring 0.898 on 13 work orders, 95% CI **[0.880, 0.938]**, whose
family measures **0.920**. The family figure sits *inside* the part's interval,
so the two are statistically indistinguishable: 0.898 versus 0.920 is what 13
work orders of sampling variation looks like, not evidence that this part mills
differently from its siblings. Giving it a separate standard would encode noise
and leave someone asking why two near-identical parts carry different times.

So `RM_RECOMMENDED` applies **one standard per family**, and a part keeps its
own value only when either:

1. the family ratio falls **outside** the part's confidence interval — the part
   demonstrably behaves differently; or
2. the family was **not itself flagged**, so there is no established alternative
   to fall back on. Substituting a figure from a pool that is not a finding
   would dilute a real part-level result with parts that do not show the
   problem — and that pool contains the part's own work orders, so it is not
   even independent.

Both candidates stay in the output beside the confidence interval, so any row
can be overridden by eye. In practice the rule leaves very few exceptions, and
those are exactly the rows worth a human look.

In [ ]:
rows = []
for op, g in wo_all.groupby("OPERATION"):
    grp = g.groupby(["FAMILY","OPERATION_SEQUENCE"])
    nrm, nparts = grp.RUN_MASTER.nunique(), grp.PART_ID.nunique()
    big = nrm[nparts >= 3]
    if len(big) < 3: continue
    rows.append((op, len(big), round(100*(big == 1).mean()), g.RUN_MASTER.nunique(),
                 round(g.HOURS_WORKED.sum()/g.HOURS_MASTER.sum(), 3)))
uni = pd.DataFrame(rows, columns=["operation","family groups (>=3 parts)",
                                  "% with ONE RUN_MASTER","distinct RUN_MASTER in op","pooled ratio"])
uni["reads as"] = np.where(uni["% with ONE RUN_MASTER"] >= 80,
                           "default -> act at FAMILY", "set per part -> act at PART")
display(uni.sort_values("% with ONE RUN_MASTER", ascending=False).head(12))

# how the part-vs-family rule actually resolves, across every operation done
import os
if os.path.exists("FTS_UPDATE_LIST.xlsx"):
    db = pd.read_excel("FTS_UPDATE_LIST.xlsx", sheet_name="02_RunUpdateList")
    print("\nWhich value RM_RECOMMENDED applies")
    for k, v in db.RECOMMENDED_BASIS.value_counts().items():
        print(f"  {k:44} {v:>4}")
    print(f"\ndistinct standards after the rule : "
          f"{db.groupby(['OPERATION','FAMILY','SEQ']).RM_RECOMMENDED.nunique().sum()}")
    print(f"if every part kept its own value  : "
          f"{db.groupby(['OPERATION','FAMILY','SEQ']).RM_SUGGESTED.nunique().sum()}")

## 12. Turning a ratio back into `RUN_MASTER`

`RUN_MASTER` is a **rate**, and hours move inversely to it. So the correction
divides:

$$\text{RUN\_MASTER}_{\text{new}} = \frac{\text{RUN\_MASTER}_{\text{old}}}{r}$$

- ratio **1.30** — the job takes 30% longer than allowed, the standard is too
  tight, so the rate must come **down**
- ratio **0.80** — the job takes only 80% of the allowed time, the standard is
  too generous, so the rate goes **up**

Dividing handles both directions automatically.

One wrinkle at family level. Where a family runs several different `RUN_MASTER`
values there is no single "current" number, so the output leaves it blank rather
than print something that would be wrong to key in. In that case you apply the
**factor** to each part's own value, which shifts the whole family
proportionally and preserves the distinctions the routing deliberately makes.

*(This inversion is specific to `PCS/HR`. `HRS/LOAD` is a duration and
multiplies instead — see below.)*

### HRS/LOAD goes the other way

For the four `HRS/LOAD` operations the master is not a rate at all but a **step
function of quantity**:

$$\text{HOURS\_MASTER} = \left\lceil \frac{\text{TOTAL\_QTY}}{\text{LOAD\_SIZE\_QTY}} \right\rceil \times \text{RUN}$$

`RUN` is hours *per load* — a duration, like `SETUP_HRS` — so the correction
**multiplies**: $\text{RUN}_{\text{new}} = \text{RUN}_{\text{old}} \times r$.
Reusing the division above would produce plausible numbers wrong in both
directions at once.

`LOAD_SIZE_QTY` is left alone. It is a physical fact — how many pieces fit in a
basket — and labour tickets carry no evidence about it; they measure elapsed
time. Correcting it from timing data would be inventing a number rather than
measuring one.

Only **single-load** work orders are tested, on `HOURS_MASTER == RUN_MASTER`.
The model assumes the second basket costs what the first did, and it does not:
single-load jobs run over their standard, multi-load jobs well under it. Fitting
$\text{HOURS\_WORKED} = a + b \times \text{loads}$ puts the marginal load
at a very different share of `RUN` depending on the operation, so no single value
of `RUN` fits both regimes.

Normalising both sides by the load count does *not* rescue the multi-load rows —
the ratio is already dimensionless, so dividing numerator and denominator by the
same number cancels and every test statistic is unchanged. It is a shape problem,
not a scale problem, and the gap is reported rather than corrected.

See `fts_load_view.py`.

## 13. What we have found so far

One workbook pair per operation, plus a consolidated update list. Below is what
those files currently contain.

In [ ]:
import glob, re, os
rows = []
for pf in sorted(glob.glob("FTS_*_PARTS.xlsx")):
    tag = re.sub(r"^FTS_|_PARTS\.xlsx$", "", pf)
    p = pd.read_excel(pf, sheet_name="05_AllUnits_WithEvidence")
    # An operation can produce no units at all - every work order a token
    # booking - so the sheet is legitimately empty and carries no operation
    # name. It contributes nothing to the totals below.
    if p.empty:
        continue
    fam = f"FTS_{tag}_FAMILY.xlsx"
    f = pd.read_excel(fam, sheet_name="01_FamilyRollup") if os.path.exists(fam) else pd.DataFrame()
    fa = f[f.FAMEX_VERDICT == "UPDATE FAMILY STANDARD"] if len(f) else f
    pa = p[p.RECOMMENDATION == "UPDATE RUN_MASTER"]
    rows.append((p.OPERATION.iat[0], len(p), int((p.N_WORK_ORDERS >= MIN_WO).sum()),
                 round(p.HOURS_WORKED.sum()/p.HOURS_MASTER.sum(), 3),
                 len(pa), len(fa), int(fa.FAMEX_PARTS.sum()) if len(fa) else 0,
                 # kept unrounded - rounding per operation and then summing
                 # drifts the total by a couple of hours
                 pa.HOURS_DELTA.sum() + (fa.FAMEX_HOURS_DELTA.sum() if len(fa) else 0),
                 # gross must be summed per ACTION, not per operation: an
                 # operation with a part running long and a family running short
                 # has two errors, not a small net one
                 pa.HOURS_DELTA.abs().sum()
                 + (fa.FAMEX_HOURS_DELTA.abs().sum() if len(fa) else 0)))
res = pd.DataFrame(rows, columns=["operation","units","testable","pooled ratio",
                                  "part actions","family actions","parts via family",
                                  "hours delta","hours gross"])
display(res.sort_values("hours delta", ascending=False)
           .style.format({"hours delta": "{:,.0f}", "hours gross": "{:,.0f}",
                          "pooled ratio": "{:.3f}"}))
print(f"TOTAL  {res['part actions'].sum() + res['family actions'].sum()} actions   "
      f"{res['part actions'].sum() + res['parts via family'].sum():,} parts   "
      f"{res['hours gross'].sum():,.0f} gross hours "
      f"({res['hours delta'].sum():,.0f} net)")

**Read the gross figure, not the net one.** The two totals above differ by an
order of magnitude, because the corrections point in both directions and largely
cancel. That cancellation is arithmetic, not good news.

A standard is consumed one part at a time — to quote it, to cost it, to load the
work centre, to judge whether the job ran well. None of those decisions ever
touches the sum. A part allowed 40% too much and another allowed 40% too little
are two wrong standards: the first quotes work too expensively and reserves
capacity nobody uses, the second produces jobs that run late and operators who
look slow. The errors compound in practice even as they cancel on paper.

The net figure would be the right one to watch only if the question were total
plant capacity for the year. It is not. The question is whether each family and
part carries a standard that describes it, and for that the honest measure of
the problem is **gross hours**.

In [ ]:
r = res.sort_values("pooled ratio")
fig, ax = plt.subplots(figsize=(7.2, .45*len(r)+1.2))
cols = np.where(r["pooled ratio"] > 1, WARM, COOL)
ax.hlines(r.operation, 1, r["pooled ratio"], color=cols, lw=2.2)
ax.scatter(r["pooled ratio"], r.operation, color=cols, s=46, zorder=3)
ax.axvline(1, color=INK, lw=1.2, ls="--")
for _, q in r.iterrows():
    off = .012 if q["pooled ratio"] > 1 else -.012
    ax.annotate(f'{q["pooled ratio"]:.3f}', (q["pooled ratio"]+off, q.operation),
                va="center", ha="left" if off > 0 else "right", fontsize=8, color=MUTED)
ax.set_xlabel("pooled ratio  (worked / master)"); ax.set_ylabel("")
ax.set_title("Where each operation sits against its own standard", fontsize=10, color=INK, loc="left")
ax.annotate("standard too generous", (.985, -.85), ha="right", fontsize=8, color=COOL, annotation_clip=False)
ax.annotate("standard too tight", (1.015, -.85), ha="left", fontsize=8, color=WARM, annotation_clip=False)
ax.grid(axis="y", visible=False); ax.margins(x=.14)
plt.tight_layout(); plt.show()

### The pattern that has emerged

Operations fall into two clearly different shapes, and it changes what you do
about them.

**Where the overhaul did real work**, standards were set per part, the pooled
ratio sits close to 1, and the job is finding the specific parts that drifted.

**Where it did not**, a single round number was stamped across every part and
every family — and that number is wrong for almost everyone, usually in the same
direction. These are the larger errors and, conveniently, the cheaper fixes:
one value corrected at family level rather than fifty individual edits.

Checking `RUN_MASTER` uniformity (§11) tells you which shape you are dealing
with before you read a single result.

## 14. What this analysis cannot tell you

Stated plainly, because these matter more than the p-values.

**A difference is not a diagnosis.** A significant result says the standard does
not match reality. It does **not** say the standard is wrong. Tooling changes,
operator mix, material variation and process drift all produce an identical
signature. Every suggestion here is a question for manufacturing engineering,
not an instruction to the ERP.

**Coverage is the real limit.** Only a minority of units have enough work orders
to be judged, and they cover well under half the hours. This is a consequence of
how much production history exists since the overhaul, and it will improve on
its own with time — not by tuning anything.

**Labour-ticket quality is assumed, not audited.** The MAD filter removes work
orders that look like mis-bookings, but nobody has verified that they are. A
meaningful share of observations get set aside this way.

**Silence is not a clean bill of health.** Dropping token bookings (§7) stops
absurd suggestions, but it does not make the underlying step visible. A step
booked at a hundredth of an hour is not a step running fast — it is a step
nobody is recording, and its standard may be badly wrong with no way to tell
from this data. Where too little is left, the step simply produces no row, and
`00_Summary` distinguishes `none - tested, holds` from `none - nothing testable`
for exactly that reason. `07_TokenBookings` lists which steps are in that state
and how much standard time the schedule reserved for them.

**No trend modelling.** One ratio per unit for the whole period. If an operation
is still climbing a learning curve, a single average understates what is
achievable.

**A standard set to observed performance is a description, not a target.** For
units running *slower* than standard, matching the observed time locks in
current performance. Whether that is desirable is a management decision this
analysis cannot make.

**Setups run by a different crew than the routing plans for are excluded.**
`HOURS_MASTER` for setup is `SETUP_HRS × LT_ROWS / USER_5` — the query emits a
copy of `SETUP_HRS / USER_5` on every labour-ticket row, and the division only
cancels when the row count equals the planned crew. It fails in both
directions: a crew of three where two booked comes out a third short, and a
one-person standard booked by two people comes out doubled.

`EMP_DISTINCT` and `LT_ROWS` were added to the extract on 13 Aug 2026 to make
this visible. Measured across the setup rows: a few percent have a crew that
does not match `USER_5` — short and over alike — and others have somebody
booking more than one ticket. Few rows fail both, so neither test can be
dropped; and the second corrupts the *ratio* rather than just the value,
because the master doubles while the labour is one person's time split in two.

Those rows are excluded from the test, not corrected. The inversion
`SETUP_HRS = HOURS_MASTER × USER_5 / LT_ROWS` is exact, but a one-person setup
performed by two people is a different job, not a fast or slow version of the
planned one — correcting it would make the number look trustworthy without
making it meaningful. They remain charged, counted and reported in the skip
figures. Cost: some performed setups leave the test, and a few testable
pools with them.

*(Two earlier estimates here — "23 rows" and "150 rows, only ever
understating" — were decoded from the rounding pattern rather than counted, and
are superseded. The decode could not see overstatement at all: at `USER_5 = 1`
the only candidate ticket count is 1.)*

*(An earlier draft of this notebook recorded a suspected bug here: that dividing
`SETUP_HRS` by `USER_5` unconditionally, while `HOURS_WORKED` is divided only
outside machining sections, inflated machining setups substantially. That was wrong on
both counts. The division is **de-duplication** — the inner query emits one row
per labour ticket, so a crew of three repeats the master value three times before
the outer `SUM`, and dividing first cancels it. The rounding proves it: crew-3
rows land on 0.1667, 0.3334, 0.5001, 0.9999 — the residue of dividing by 3 at
`NUMERIC(19,4)` — where crew-1 and crew-2 rows sit on clean halves and quarters.
And the asymmetry cannot bite anyway: on every machining setup row the
effective divisor is 1 — stored as 1 or NULL, which `ISNULL(O.USER_5, 1)`
resolves to 1 — so the division is a no-op there. See DESIGN.md for the full
account.)*

## 15. Running it

Three commands per operation, in order. The third consolidates every operation
analysed so far and **must be re-run each time one is added**.

```bash
python fts_analysis.py    "S LUCIDARE"   # part level   -> FTS_<OP>_PARTS.xlsx
python fts_family_view.py "S LUCIDARE"   # family level -> FTS_<OP>_FAMILY.xlsx
python fts_update_list.py                # consolidated -> FTS_UPDATE_LIST.xlsx
                                           # run it LAST - it folds in setup and HRS/LOAD too
```

Then re-run this notebook so its figures catch up.

After **any** change to either analysis script, re-run every operation. A stale
workbook looks completely normal and says nothing about being out of date — this
has already caused one wrong answer.

```bash
for OP in "S FRESATURA" "S LUCIDARE DRAG" "S LIQ PENETRANT" "S LUCIDARE"; do
  python fts_analysis.py "$OP"; python fts_family_view.py "$OP"
done
python fts_update_list.py
```

### Reading the output

| Sheet | Use it for |
|---|---|
| `_PARTS` → `02_Table_Tested` | the working table, one row per testable part |
| `_PARTS` → `03_Actions` | part-level changes with full statistical evidence |
| `_PARTS` → `06_ActionDetail` | work-order-by-work-order backing, for challenges |
| `_FAMILY` → `01_FamilyRollup` | **every** family, including those with no testable part |
| `_FAMILY` → `02_FamilyActions` | family-level changes |
| `FTS_UPDATE_LIST.xlsx` | everything - all three targets, both levels, all operations |
| `UPDATE_LIST` → `07_TokenBookings` | steps closed without a duration — what the analysis cannot see |

### Keeping this notebook honest

Edit it directly — it is a normal notebook, not generated from anything. Every
figure is computed live from `fts.parquet` and the workbooks, so the only
maintenance is **re-running all cells** after new results land.

If the flow changes — a new filter, a different threshold, a new test — update
the prose here in the same pass as the script change. The explanation and the
code are only as trustworthy as the last time someone reconciled them.